# Phase 4 Hyperparameter Tuning (Price + Sentiment)

Optional tuning notebook for Ablation 2.

Outputs:
- `results/phase4_tuning/price_sentiment_tuning_trials.csv`
- `results/phase4_tuning/best_params_price_sentiment.json`


In [1]:
# Optional installs
# %pip install gymnasium stable-baselines3


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

from phase2_trading_env import make_env_from_processed, DEFAULT_TICKERS

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv


In [3]:
TRAIN_PATH = "processed/train_dataset.csv"
FEATURE_CONFIG_PATH = "processed/feature_config.json"
OUT_DIR = Path("results/phase4_tuning")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TRIALS_CSV = OUT_DIR / "price_sentiment_tuning_trials.csv"
BEST_JSON = OUT_DIR / "best_params_price_sentiment.json"

N_TRIALS = 12
SEEDS = [7, 42]
TIMESTEPS_PER_TRIAL = 20_000
VAL_SPLIT_DATE = "2021-01-01"

cfg = json.loads(Path(FEATURE_CONFIG_PATH).read_text())
features = cfg["price_features"] + cfg["sentiment_features"]
full_train = pd.read_csv(TRAIN_PATH, parse_dates=["date"]).sort_values(["date","ticker"])
sub_train = full_train[full_train["date"] < pd.Timestamp(VAL_SPLIT_DATE)].copy()
sub_val = full_train[full_train["date"] >= pd.Timestamp(VAL_SPLIT_DATE)].copy()


In [4]:
def build_env_from_df(df):
    tmp = OUT_DIR / "_tmp_split.csv"
    df.to_csv(tmp, index=False)
    return make_env_from_processed(
        dataset_path=tmp,
        feature_config_path=FEATURE_CONFIG_PATH,
        tickers=DEFAULT_TICKERS,
        feature_columns_override=features,
        initial_cash=1_000_000.0,
        transaction_cost=0.001,
        trade_fraction=0.10,
        use_event_scaling=False,
    )


def run_backtest(model, env, seed=42):
    obs, info = env.reset(seed=seed)
    done = False
    values=[info["portfolio_value"]]
    daily=[]
    while not done:
        action,_=model.predict(obs, deterministic=True)
        obs,reward,done,_,step=env.step(action)
        values.append(float(step["portfolio_value"]))
        daily.append(float(step["daily_return"]))
    return pd.Series(values), pd.Series(daily)


def sharpe(d):
    s=d.std(ddof=1)
    if s<=1e-12: return 0.0
    return float((d.mean()/s)*np.sqrt(252))


def mdd(v):
    peak=v.cummax()
    return float((v/peak-1.0).min())


In [5]:
rng = np.random.default_rng(42)

def sample_params():
    return {
        "learning_rate": float(10 ** rng.uniform(-4.5, -3.0)),
        "n_steps": int(rng.choice([512, 1024, 2048, 4096])),
        "batch_size": int(rng.choice([64, 128, 256, 512])),
        "n_epochs": int(rng.choice([5, 10])),
        "gamma": float(rng.choice([0.97, 0.99, 0.995])),
        "gae_lambda": float(rng.choice([0.90, 0.95, 0.98])),
        "ent_coef": float(rng.choice([0.0, 0.001, 0.005, 0.01])),
        "clip_range": float(rng.choice([0.1, 0.2, 0.3])),
        "vf_coef": float(rng.choice([0.25, 0.5, 0.75])),
    }

rows=[]
for trial in range(N_TRIALS):
    params=sample_params()
    per_seed=[]
    for seed in SEEDS:
        env_train=DummyVecEnv([lambda: build_env_from_df(sub_train)])
        model=PPO("MlpPolicy", env_train, verbose=0, seed=seed, **params)
        model.learn(total_timesteps=TIMESTEPS_PER_TRIAL)
        env_val=build_env_from_df(sub_val)
        values,daily=run_backtest(model, env_val, seed=seed)
        ret=float(values.iloc[-1]/values.iloc[0]-1.0)
        s=sharpe(daily)
        draw=mdd(values)
        score=s + 2.0*ret - 0.5*abs(draw)
        per_seed.append((score,s,ret,draw))
    row={"trial":trial, **params,
         "score_mean": float(np.mean([x[0] for x in per_seed])),
         "sharpe_mean": float(np.mean([x[1] for x in per_seed])),
         "cumret_mean": float(np.mean([x[2] for x in per_seed])),
         "maxdd_mean": float(np.mean([x[3] for x in per_seed]))}
    rows.append(row)
    print(f"trial={trial:02d} score={row['score_mean']:.4f}")

trials_df=pd.DataFrame(rows).sort_values("score_mean", ascending=False).reset_index(drop=True)
trials_df.to_csv(TRIALS_CSV, index=False)
trials_df.head(10)


trial=00 score=-11.0403
trial=01 score=-7.9538
trial=02 score=-3.1513
trial=03 score=-7.2038
trial=04 score=-5.5390
trial=05 score=-4.3973
trial=06 score=-3.4396
trial=07 score=-2.8808
trial=08 score=-2.8347
trial=09 score=-15.8049
trial=10 score=-6.0613
trial=11 score=-13.0224


,trial,learning_rate,n_steps,batch_size,n_epochs,gamma,gae_lambda,ent_coef,clip_range,vf_coef,score_mean,sharpe_mean,cumret_mean,maxdd_mean
0,8,0.000143,512,512,10,0.995,0.90,0.001,0.3,0.75,-2.834719,-3.107488,0.178964,-0.170318
1,7,0.000061,1024,64,10,0.990,0.90,0.000,0.2,0.75,-2.880754,-3.679714,0.434147,-0.138668
2,2,0.000114,512,512,10,0.990,0.95,0.010,0.2,0.50,-3.151313,-3.205790,0.092420,-0.260723
3,6,0.000414,1024,512,5,0.970,0.98,0.001,0.1,0.50,-3.439554,-3.722702,0.170034,-0.113841
4,5,0.000062,1024,128,5,0.970,0.95,0.000,0.3,0.75,-4.397338,-4.623010,0.144964,-0.128513
5,4,0.000434,2048,128,5,0.995,0.95,0.010,0.3,0.75,-5.538996,-5.702662,0.114652,-0.131278
6,10,0.000063,4096,64,10,0.995,0.98,0.005,0.2,0.75,-6.061258,-6.340830,0.167409,-0.110494
7,3,0.000069,512,256,10,0.970,0.98,0.010,0.1,0.50,-7.203833,-7.595828,0.230446,-0.137792
8,1,0.000919,2048,512,10,0.995,0.95,0.000,0.3,0.50,-7.953820,-8.172229,0.146788,-0.150334
9,0,0.000458,2048,128,5,0.995,0.90,0.005,0.1,0.25,-11.040334,-11.273865,0.142653,-0.103551


In [6]:
best = trials_df.iloc[0].to_dict()
keys = ["learning_rate","n_steps","batch_size","n_epochs","gamma","gae_lambda","ent_coef","clip_range","vf_coef"]
payload = {
    "source": "phase4_hyperparam_tuning.ipynb",
    "objective": "score_mean = sharpe + 2*cumret - 0.5*|max_drawdown|",
    "n_trials": int(N_TRIALS),
    "seeds": SEEDS,
    "timesteps_per_trial": int(TIMESTEPS_PER_TRIAL),
    "val_split_date": VAL_SPLIT_DATE,
    "best_params": {k: best[k] for k in keys},
    "best_row": best,
}
with open(BEST_JSON, "w") as f:
    json.dump(payload, f, indent=2)
print("Saved:", BEST_JSON)
print(json.dumps(payload["best_params"], indent=2))


Saved: results/phase4_tuning/best_params_price_sentiment.json
{
  "learning_rate": 0.00014312907952658717,
  "n_steps": 512.0,
  "batch_size": 512.0,
  "n_epochs": 10.0,
  "gamma": 0.995,
  "gae_lambda": 0.9,
  "ent_coef": 0.001,
  "clip_range": 0.3,
  "vf_coef": 0.75
}
